# PID-regeling vanuit Python

Upload eerst `arduino/PID_Python_aansturing/PID_Python_aansturing.ino`. Sluit de Serial Monitor en Serial Plotter. De Arduino voert de PID-regeling uit; Python stelt de parameters in en slaat de meting op.

In [ ]:
from closed_loop.meting import controleer_sensor, voer_pid_meting_uit
import matplotlib.pyplot as plt

com_poort = "COMX"  # Vul de poort van jouw Arduino in.

### 1. Sensor controleren

Laat de motorvoeding uit. De werkhoek is `90 - sensorhoek`; controleer de nulstand en richting voor jouw opstelling. De sketch accepteert werkhoeken van -10 tot 80 graden. Na de controle toont deze cel de ontvangen hoeken.

In [ ]:
controle, _ = controleer_sensor(com_poort=com_poort)
if controle:
    tijden, doelen, hoeken, pwm_waarden = zip(*controle)
    plt.plot(tijden, hoeken)
    plt.xlabel("Tijd (s)")
    plt.ylabel("Werkhoek (graden)")
    plt.grid()
    plt.show()

### 2. Instellen en meten

Begin met P: `ki = 0` en `kd = 0`. Kies met de begeleider een haalbare doelhoek, versterking en PWM-grens op basis van open loop. De waarden hieronder zijn beginwaarden. Schakel daarna de gecontroleerde motorvoeding in. Deze cel start de regeling direct na de communicatiecontrole, zonder stabilisatietijd. Onderbreek de cel om eerder te stoppen.

In [ ]:
doelhoek = 40.0    # graden ten opzichte van de ingestelde nulstand
kp = 2.0
ki = 0.0
kd = 0.0
max_pwm = 120     # bovengrens; kies een lagere waarde indien nodig
meetduur = 30.0   # seconden

metingen, csv_pad = voer_pid_meting_uit(
    com_poort=com_poort,
    doelhoek=doelhoek,
    kp=kp, ki=ki, kd=kd,
    max_pwm=max_pwm,
    meetduur=meetduur,
)

### 3. Resultaat bekijken

Meetpunten staan in `data/` als CSV; een JSON-bestand met dezelfde naam bewaart de gekozen instellingen en de afloopstatus. Verander voor PI alleen `ki`; onderzoek daarna voor PID ook `kd`. Laat de arm voor iedere proef tot rust komen.

In [ ]:
if metingen:
    tijden, doelen, hoeken, pwm_waarden = zip(*metingen)
    fig, ax = plt.subplots(2, 1, sharex=True, figsize=(8, 6))
    ax[0].plot(tijden, hoeken, label="Gemeten hoek")
    ax[0].plot(tijden, doelen, "--", label="Doelhoek")
    ax[0].set_ylabel("Hoek (graden)")
    ax[0].legend()
    ax[1].plot(tijden, pwm_waarden)
    ax[1].set_ylabel("PWM")
    ax[1].set_xlabel("Tijd sinds start (s)")
    for a in ax:
        a.grid()
    fig.tight_layout()
    plt.show()